# Training loop for mini-gLM

## Load data

Import packages and configure root.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys, pathlib, subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
!pip install pyfaidx
from pyfaidx import Fasta
from pathlib import Path
from huggingface_hub import hf_hub_download
from src.transformer import MoETransformer
from src.tokenize import BPETokenizer
from torch.utils.data import DataLoader, Dataset, BatchSampler
from torch.nn.utils.rnn import pad_sequence

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/mini-gLM.git"
repo_dir = Path("mini-gLM")
if COLAB:
    root = Path("/content/mini-gLM")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

# Use GPU if available
torch.manual_seed(111)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Model architecture

In [ ]:
class DenseGLM(nn.Module):
    def __init__(self, vocab_size, num_blocks, d_model, num_heads, h_dim, num_experts, top_k, p_drop):

        self.vocab_size = vocab_size
        self.num_blocks = num_blocks
        self.d_model = d_model
        self.num_heads = num_heads
        self.h_dim = h_dim
        self.num_experts = num_experts
        self.top_k = top_k
        self.p_drop = p_drop

        self.embedding = nn.Embedding(
            num_embeddings = self.vocab_size,
            embedding_dim = self.d_model
        )

        self.moe_transformer = MoETransformer(
            d_model = self.d_model,
            num_heads = self.num_heads,
            h_dim = self.h_dim,
            num_experts = self.num_experts,
            top_k = self.top_k,
            p_drop = self.p_drop
        )

        self.model = nn.ModuleList([
            self.moe_transformer for _ in range(num_blocks)
        ])

        self.final_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):

        x = self.model(x)
        logits = self.final_head(x)

        return logits

## Training

Configure the MLM training loop.
- Include W&B experiment tracking
- Easy parameter loading
- Mixed precision tuning
- Muon optimizer
- Figure out how to move the attention mask into the model?

In [ ]:
from src.train import hg38Data, DynamicBatchSampler, MLMCollator

# Define model params dictionary
model_params = {
    "vocab_size": 512,
    "num_blocks": 5,  # Change
    "d_model": 1,     # Change
    "num_heads": 1,   # Change
    "h_dim": 1,       # Change
    "num_experts": 1, # Change
    "top_k": 2,
    "p_drop": 0.1
}

def train_glm(epochs, lr, model_params):

    # Load train and val data
    train_data = hg38Data()
    val_data  = hg38Data()

    train_loader = DataLoader(
        dataset = train_data, 
        batch_sampler = DynamicBatchSampler(),
        collate_fn = MLMCollator      
    )
    val_loader = DataLoader(
        val_data,
        batch_sampler = DynamicBatchSampler(),
        collate_fn = MLMCollator
    )

    model = DenseGLM(**model_params)
    model.to(device).float()

    # Muon optimizer
    optim = torch.optim.Muon(model.parameters(), lr = lr)

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        # Train loss
        epoch_train_loss = 0.0

        for batch_items in train_loader:
        
            # Store batch items (match with output of collate_fn)
            batch = batch_items["batch"].to(device).float()
            labels = batch_items["labels"].to(device).float()
            predict_mask = batch_items["predict_mask"].to(device)
            attention_mask = batch_items["attention_mask"].to(device)

            # Generate predicted tokens and apply prediction mask to labels
            logits = model(batch)
            labels[~predict_mask] = -100

            # Initiate CE loss
            loss_fn = nn.CrossEntropyLoss(ignore_index = -100)

            # Calculate CE loss
            loss = loss_fn(
                logits.view(-1, logits.shape(-1))  # [B*L, vocab_size]
                labels.view(-1)                    # [B*L]
            )

            # clear gradient, backprop, update params
            optim.zero_grad(); loss.backward(); optim.step()

            # Total loss = average batch loss * batch size
            epoch_train_loss += loss.item() * logits.size(0)
        
        # Compute average
        train_losses.append(epoch_train_loss / len(train_data))
    
        epoch_val_loss = 0.0

        for batch_items in val_loader:

            with torch.no_grad():

                # Store batch items
                batch = batch_items["batch"].to(device).float()
                labels = 

                # Generate predicted tokens

                # Calculate CE loss

                # clear gradient, backprop, update params

                # Total loss = average batch loss * batch size

        # Copmpute average loss
        val_losses.append(epoch_val_loss / len(val_data))
        # Return loss every 10


        # Rep[ort train and val losses
    
        print("Training complete!")